## Imports

In [ ]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../../'))

import EXP_neuro_fuzzy_toolbox as nft

In [2]:
import numpy as np

In [3]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Multiclass Classification

## Data

In [23]:
from ucimlrepo import fetch_ucirepo

In [24]:
heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features
y = heart_disease.data.targets

In [25]:
X = X.fillna(value=0)

In [29]:
y[y["num"] >= 1] = 1
y

/tmp/ipykernel_18009/4006239884.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y[y["num"] >= 1] = 1
/tmp/ipykernel_18009/4006239884.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y[y["num"] >= 1] = 1


,num
0,0
1,1
2,1
3,0
4,0
...,...
298,1
299,1
300,1
301,1


In [30]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

In [31]:
unique_classes, counts = np.unique(y_train.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1]
[115  97]


In [32]:
unique_classes, counts = np.unique(y_test.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1]
[49 42]


In [33]:
scaler = MinMaxScaler(feature_range=(0, 1))

x_train = scaler.fit_transform(x_train)

x_test = scaler.transform(x_test)

In [34]:
x_train = torch.from_numpy(x_train)
x_test = torch.from_numpy(x_test)
y_train = torch.from_numpy(y_train.values).squeeze()
y_test = torch.from_numpy(y_test.values).squeeze()

In [35]:
y_train.dtype

torch.int64

In [36]:
x_train = x_train.type(torch.float32)
x_test = x_test.type(torch.float32)

In [37]:
loader = data.DataLoader(data.TensorDataset(x_train, y_train), batch_size = 4, shuffle = True)
x_trainset = loader.dataset.tensors[0]
y_trainset = loader.dataset.tensors[1]

In [38]:
y_trainset.dtype

torch.int64

## Model & Training

In [50]:
model = nft.rule_reduced_ANFIS(
    input_size = 13,
    num_mfs = 15,
    outputs = 2,
    output_type='softmax',
    dtype=torch.float32
)

In [52]:
loss_fn = nn.functional.cross_entropy

optimizer = torch.optim.AdamW
params = {'lr': 0.001, 'weight_decay': 0.01}
#optimizer = torch.optim.Adam
#params = {'lr': 0.005}
#optimizer = torch.optim.SGD
#params = {'lr': 0.001, 'momentum': 0.9}

early_stopping = nft.EarlyStopping(
    patience=50, 
    delta=0.01
)

In [53]:
trainer = nft.Hybrid_learning_algorithm(
    epochs=500,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    validation=0.3,
    early_stopping=early_stopping
)

In [54]:
trainer(model, loader, verbose=True)

Epoch:   1/500 - loss: 0.504216 - validation loss: 0.541710
Epoch:   2/500 - loss: 0.735913 - validation loss: 0.747839
Epoch:   3/500 - loss: 0.691566 - validation loss: 0.685079
Epoch:   4/500 - loss: 0.627921 - validation loss: 0.612936
Epoch:   5/500 - loss: 0.623367 - validation loss: 0.631863
Epoch:   6/500 - loss: 0.594057 - validation loss: 0.633242
Epoch:   7/500 - loss: 0.597776 - validation loss: 0.616439
Epoch:   8/500 - loss: 0.583118 - validation loss: 0.607408
Epoch:   9/500 - loss: 0.559487 - validation loss: 0.594207
Epoch:  10/500 - loss: 0.521374 - validation loss: 0.533694
Epoch:  11/500 - loss: 0.527458 - validation loss: 0.567546
Epoch:  12/500 - loss: 0.494265 - validation loss: 0.509333
Epoch:  13/500 - loss: 0.483232 - validation loss: 0.498831
Epoch:  14/500 - loss: 0.458279 - validation loss: 0.478399
Epoch:  15/500 - loss: 0.454562 - validation loss: 0.472499
Epoch:  16/500 - loss: 0.456736 - validation loss: 0.473717
Epoch:  17/500 - loss: 0.444035 - valida

In [55]:
train_measures = nft.get_measures(model, x_trainset, y_trainset)
for measure in train_measures:
    print(measure + ':', train_measures[measure])

Accuracy: 0.8820754716981132
Precision: 0.8831039282400248
Recall: 0.8820754716981132
F1: 0.881609829593083
Confusion Matrix: [[106   9]
 [ 16  81]]


In [56]:
test_measures = nft.get_measures(model, x_test, y_test)
for measure in test_measures:
    print(measure + ':', test_measures[measure])

Accuracy: 0.7912087912087912
Precision: 0.7909943714821764
Recall: 0.7912087912087912
F1: 0.7910055138970803
Confusion Matrix: [[40  9]
 [10 32]]
